In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 96.2 MB/s eta 0:00:00


In [ ]:
import zipfile
import json
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import faiss
import time
from pathlib import Path
from google.colab import drive

In [ ]:
drive.mount("/content/drive")

SPLIT_PATH = Path("/content/drive/MyDrive/thesis_results/SCOTBESS_splits/SCOTBESS_FULL_ANNOTATED_TERRA_LOW_SPLIT.csv")
OUTPUT_DIR = Path("/content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_embeddings_final")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

scotbess_df = pd.read_csv(SPLIT_PATH)
scotbess_df["label_list"] = scotbess_df["labels"].apply(json.loads)

print(scotbess_df.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(1675, 8)


In [ ]:
scotbess_df_train = scotbess_df[scotbess_df["split"] == "train"].copy()
scotbess_df_val = scotbess_df[scotbess_df["split"] == "validation"].copy()
scotbess_df_test = scotbess_df[scotbess_df["split"] == "test"].copy()

print(f"Train: {len(scotbess_df_train)}")
print(f"Validation: {len(scotbess_df_val)}")
print(f"Test: {len(scotbess_df_test)}")

Train: 1340
Validation: 165
Test: 170


In [ ]:
scotbess_df_train["final_masked_text"].str.split().str.len().describe()


,final_masked_text
count,1340.000000
mean,442.158209
std,761.337197
min,1.000000
25%,62.000000
50%,161.000000
75%,461.000000
max,6591.000000


In [ ]:
scotbess_df_val["final_masked_text"].str.split().str.len().describe()


,final_masked_text
count,165.000000
mean,464.290909
std,783.894950
min,4.000000
25%,57.000000
50%,154.000000
75%,479.000000
max,5714.000000


In [ ]:
print(scotbess_df_train[["final_masked_text", "label_list"]].isna().sum())
print(scotbess_df_val[["final_masked_text", "label_list"]].isna().sum())
print(scotbess_df_test[["final_masked_text", "label_list"]].isna().sum())


final_masked_text    0
label_list           0
dtype: int64
final_masked_text    0
label_list           0
dtype: int64
final_masked_text    0
label_list           0
dtype: int64


In [ ]:
#standard top-k selection,  identical in logic to the AAPD implementation
def standard_top_k_selection(pool_indices, pool_distances, target_k=5):
    selected_indices = [int(idx) for idx in pool_indices[:target_k]]
    selected_scores = [float(score) for score in pool_distances[:target_k]]

    return selected_indices, selected_scores

In [ ]:
MODEL_NAME = "BAAI/bge-m3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def sync_gpu():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

experiment_start = time.perf_counter()

# model initialization; kept parallel to AAPD
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model.max_seq_length = 8192 #BGE longest supported maximum; the longest representation in SCOTBESS has 6591 words


scotbess_df_train = scotbess_df_train.reset_index(drop=True)
scotbess_df_test = scotbess_df_test.reset_index(drop=True)
scotbess_df_val = scotbess_df_val.reset_index(drop=True)

train_texts = scotbess_df_train["final_masked_text"].tolist()
test_texts = scotbess_df_test["final_masked_text"].tolist()
val_texts = scotbess_df_val["final_masked_text"].tolist()

#encode training set
sync_gpu()
start = time.perf_counter()

train_embeddings = model.encode(
    train_texts,
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

sync_gpu()
train_encoding_time = time.perf_counter() - start

# index
start = time.perf_counter()

dimension = train_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(train_embeddings)

index_build_time = time.perf_counter() - start

np.save(OUTPUT_DIR / "scotbess_train_bge_m3_embeddings.npy", train_embeddings)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/168 [00:00<?, ?it/s]

In [ ]:
POOL_SIZE = 20
TARGET_K = 5    #final number of examples

def encode_and_search(query_df):
    query_texts = query_df["final_masked_text"].tolist()

    sync_gpu()
    start = time.perf_counter()

    query_embeddings = model.encode(query_texts, batch_size=8, show_progress_bar=True,normalize_embeddings=True).astype("float32")

    sync_gpu()
    encoding_time = time.perf_counter() - start

    sync_gpu()
    start_search = time.perf_counter()

    distances_pool, indices_pool = index.search(query_embeddings, POOL_SIZE)

    sync_gpu()
    search_time = time.perf_counter() - start_search

    return distances_pool, indices_pool, encoding_time, search_time

In [ ]:
def build_retrieval_results(query_df, distances_pool, indices_pool):
    retrieval_results = {}

    start_selection = time.perf_counter()

    for i in range(len(indices_pool)):
        sel_indices, sel_scores = standard_top_k_selection(indices_pool[i], distances_pool[i], target_k=TARGET_K)


  #creating a json which contains not only the indexes and similarity, but also the label names and texts
  #for audit only
        retrieval_results[i] = {
            "query_labels": query_df.iloc[i]["label_list"],
            "retrieved_train_indices": sel_indices,
            "similarity_scores": sel_scores,
            "retrieved_train_labels": [
                scotbess_df_train.iloc[idx]["label_list"] for idx in sel_indices],
            "retrieved_train_texts": [
                scotbess_df_train.iloc[idx]["final_masked_text"] for idx in sel_indices]}

    selection_time = time.perf_counter() - start_selection

    return retrieval_results, selection_time

In [ ]:
#val
val_distances_pool, val_indices_pool, val_encoding_time, val_search_time = encode_and_search(scotbess_df_val)
val_retrieval_results_topk, val_selection_time_topk = build_retrieval_results(query_df=scotbess_df_val, distances_pool=val_distances_pool, indices_pool=val_indices_pool)

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

In [ ]:
#test
test_distances_pool, test_indices_pool, test_encoding_time, test_search_time = encode_and_search(scotbess_df_test)
test_retrieval_results_topk, test_selection_time_topk = build_retrieval_results(query_df=scotbess_df_test, distances_pool=test_distances_pool, indices_pool=test_indices_pool)

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

In [ ]:
retrieval_files = {OUTPUT_DIR / "scotbess_validation_top_k_retrieval_results.json": val_retrieval_results_topk,OUTPUT_DIR / "scotbess_test_top_k_retrieval_results.json": test_retrieval_results_topk}

for output_path, retrieval_results in retrieval_files.items():
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(retrieval_results, f, ensure_ascii=False, indent=2)

In [ ]:
timing_results = {
    "model_name": MODEL_NAME,
    "dataset": "Scot-BESS",
    "device": DEVICE,
    "max_seq_length": 8192,
    "batch_size": 8,
    "pool_size": POOL_SIZE,
    "n_retrieved": TARGET_K,
    "train_encoding_time_sec": train_encoding_time,
    "index_build_time_sec": index_build_time,
    "validation_encoding_time_sec": val_encoding_time,
    "validation_faiss_search_time_sec": val_search_time,
    "validation_topk_selection_time_sec": val_selection_time_topk,
    "validation_faiss_ms_per_query": (val_search_time / len(scotbess_df_val)) * 1000,
    "test_encoding_time_sec": test_encoding_time,
    "test_faiss_search_time_sec": test_search_time,
    "test_topk_selection_time_sec": test_selection_time_topk,
    "test_faiss_ms_per_query": (test_search_time / len(scotbess_df_test)) * 1000,
    "total_experiment_time_sec": time.perf_counter() - experiment_start}

with open(OUTPUT_DIR / "scotbess_retrieval_timing_results.json", "w", encoding="utf-8") as f:
    json.dump(timing_results, f, indent=2)

print("\nTiming results:")
for key, value in timing_results.items():
    print(f"{key}: {value}")


Timing results:
model_name: BAAI/bge-m3
dataset: Scot-BESS
device: cuda
max_seq_length: 8192
batch_size: 8
pool_size: 20
n_retrieved: 5
train_encoding_time_sec: 282.2136654630001
index_build_time_sec: 0.021518332000141527
validation_encoding_time_sec: 58.013888783000084
validation_faiss_search_time_sec: 0.06739514700007021
validation_topk_selection_time_sec: 0.06405216400003155
validation_faiss_ms_per_query: 0.40845543636406184
test_encoding_time_sec: 47.6376676320001
test_faiss_search_time_sec: 0.011631488999910289
test_topk_selection_time_sec: 0.09438700400005473
test_faiss_ms_per_query: 0.06842052352888404
total_experiment_time_sec: 493.9904936219998


In [ ]:
#sanity check
#do the retreived samples actually contain the labels of the test set?
#calculating coverage metric (the number of queries for which all labels are covered across the retreived texts)

def run_label_audit(json_path, dataset_name):
    with open(json_path, "r", encoding="utf-8") as f:
        results = json.load(f)

    full_coverage_count = 0
    recall_scores = []
    similarity_scores = []

    for query_id, data in results.items():
        target_labels = set(data["query_labels"])
        #flatten the list-of-lists containing the labels from the top5k neighbours
        pooled_candidate_labels = set(label for neighbor_labels in data["retrieved_train_labels"] for label in neighbor_labels)

        #Metric 1: Full Coverage (all target labels found in the top-k)
        if target_labels.issubset(pooled_candidate_labels):
            full_coverage_count += 1

        #Metric 2: proportion of gold labels covered by the retrieved examples (e.g., the query had 3 labels, but only 1 of them was present in the retrieved set)
        intersected = target_labels.intersection(pooled_candidate_labels)
        recall_scores.append(len(intersected) / len(target_labels) if target_labels else 1.0)

        #Metric 3: Cosine Similarity of the retreived examples
        similarity_scores.extend(data["similarity_scores"])

    total_queries = len(results)
    stats = {
        "Dataset": dataset_name,
        "Number of Queries": total_queries,
        "Full Label Coverage@5": f"{(full_coverage_count / total_queries) * 100:.2f}%",
        "Mean Label Recall @5": f"{np.mean(recall_scores) * 100:.2f}%",
        "Mean Cosine Similarity @5": f"{np.mean(similarity_scores):.2f}"}

    return stats

val_stats_topk = run_label_audit(OUTPUT_DIR / "scotbess_validation_top_k_retrieval_results.json", "Validation Top-K")
test_stats_topk = run_label_audit(OUTPUT_DIR / "scotbess_test_top_k_retrieval_results.json", "Test Top-K")

print("Validation set")
print(json.dumps(val_stats_topk, indent=2))
print("Test set")
print(json.dumps(test_stats_topk, indent=2))

Validation set
{
  "Dataset": "Validation Top-K",
  "Number of Queries": 165,
  "Full Label Coverage@5": "74.55%",
  "Mean Label Recall @5": "94.77%",
  "Mean Cosine Similarity @5": "0.81"
}
Test set
{
  "Dataset": "Test Top-K",
  "Number of Queries": 170,
  "Full Label Coverage@5": "80.00%",
  "Mean Label Recall @5": "94.21%",
  "Mean Cosine Similarity @5": "0.80"
}


In [ ]:
# creating input files which will be used by the LLM (not containing query labels)
def create_llm_data_file(query_df, retrieval_results, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for i, item in retrieval_results.items():
            i = int(i)

            retrieved_examples = []

            for idx,  labels, text in zip(
                item["retrieved_train_indices"],
                item["retrieved_train_labels"],
                item["retrieved_train_texts"]):
                retrieved_examples.append({
                    "train_index": int(idx),
                    "labels": labels,
                    "text": text})

            row = {
                "query_id": i,
                "target_text": query_df.iloc[i]["final_masked_text"],
                "retrieved_examples": retrieved_examples}

            f.write(json.dumps(row, ensure_ascii=False) + "\n")

create_llm_data_file(query_df=scotbess_df_val,retrieval_results=val_retrieval_results_topk,output_path=OUTPUT_DIR / "scotbess_validation_top_k_llm_input_data.jsonl")

create_llm_data_file(query_df=scotbess_df_test,retrieval_results=test_retrieval_results_topk,output_path=OUTPUT_DIR / "scotbess_test_top_k_llm_input_data.jsonl")


In [ ]:
#files for later LLM evaluation (with gold labels)
def create_gold_label_file(query_df, output_path):
    gold_labels = {}

    for i in range(len(query_df)):
        gold_labels[int(i)] = query_df.iloc[i]["label_list"]

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(gold_labels, f, ensure_ascii=False, indent=2)

In [ ]:
create_gold_label_file(query_df=scotbess_df_val, output_path=OUTPUT_DIR / "scotbess_validation_gold_labels.json")
create_gold_label_file(query_df=scotbess_df_test, output_path=OUTPUT_DIR / "scotbess_test_gold_labels.json")